# Lexsi GRPO rollout diagnostics (Colab)

Diagnostic-only path for reviewer checks when the lab RTX 5060 Ti is unavailable.

**Does not** replace the pinned study host. `--allow-any-gpu` / `LEXSI_ALLOW_ANY_GPU=1` is required on Colab and must not be used for claimed study evidence.

See `docs/COLAB_ROLLOUT_DIAGNOSTICS.md`.

## 0. Runtime

Runtime → Change runtime type → **GPU** (T4 is enough for a few Tiny Aya bf16 rollouts).

In [ ]:
# Clone + thin train deps (AgentTune full lock not required for dumps)
import os
from pathlib import Path

REPO = Path("/content/Hackathons")
if not (REPO / "lexsi").exists():
    !git clone --depth 1 https://github.com/AriCommits/Hackathons.git {REPO}
%cd {REPO}/lexsi
!pip -q install -e ".[train]"
print("cwd", Path.cwd())

In [ ]:
# HF secret (Colab: Secrets panel → HF_TOKEN)
from google.colab import userdata

token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
print("HF_TOKEN set:", bool(token))

## 1–3. CPU checks (no GPU, no adapter)

- Prompt train vs eval diff
- Reward ledger meaning for −0.1
- Fixture multi-turn dumps (includes one parse_error → −0.1)

**Code finding:** training rollouts are multi-turn (`BehaviorToolRollout`), same as eval. −0.1 is first-decision failure (k=0, invalid_attempts=1), not single-turn TRL.

In [ ]:
!python scripts/diagnose_grpo_rollouts.py --prompt-diff
!python scripts/diagnose_grpo_rollouts.py --explain-reward
!python scripts/diagnose_grpo_rollouts.py --fixture-rollouts --count 4 --output /content/fixture_rollouts.json

## 4. Live GPU dumps (Tiny Aya ± e2 adapter)

Upload `checkpoint-150` from the study host (or mount Drive) to `/content/e2-checkpoint-150`.
Without the adapter, dumps still run on the base model for format/truncation probes.

Colab **does not** have UUID `134c34ef-c155-b001-4e3a-b9134b072c21` → must pass `--allow-any-gpu`.

In [ ]:
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

ADAPTER = "/content/e2-checkpoint-150"
adapter_flag = f"--adapter {ADAPTER}" if Path(ADAPTER).is_dir() else ""
print("adapter", ADAPTER, "exists", Path(ADAPTER).is_dir())

!python scripts/diagnose_grpo_rollouts.py --live-rollouts --allow-any-gpu \
  {adapter_flag} --count 4 --output /content/live_rollouts.json

In [ ]:
import json
from pathlib import Path

path = Path("/content/live_rollouts.json")
if not path.exists():
    path = Path("/content/fixture_rollouts.json")
payload = json.loads(path.read_text())
for i, row in enumerate(payload["rollouts"]):
    ledger = row["ledger"]
    print("=" * 60)
    print(f"rollout {i} reason={row['completion_reason']} tokens={row['completion_token_count']}")
    print("prompt:", row["prompt"][:200])
    print("completion:", row["decoded_completion"][:500])
    print("ledger:", {k: ledger[k] for k in (
        "reward", "invalid_attempts", "verified_success", "completion_reason",
        "duplicate_calls", "excess_calls",
    )})
print("truncation_stats:", payload.get("truncation_stats"))

## Unattended one-shot (close laptop after start)

Run **only this next cell** after Runtime → GPU. It mounts Drive, installs, dumps 4 live rollouts, writes `MyDrive/lexsi/grpo_diag/live_rollouts.json` + a compact summary with a `completion_reason` Counter, then prints `DONE`.

**Caveat:** free Colab may still stop if the browser tab dies. Leave the tab open on a phone if you can; otherwise prefer Colab Pro. There is no email callback — recover results from Drive later.

Optional: put e2 at `MyDrive/lexsi/e2-checkpoint-150/` first. Without it, base Tiny Aya still probes format/truncation/template.

In [ ]:
# Unattended GRPO rollout dump → Google Drive (safe to leave after STARTING prints)
import json, os, subprocess, sys
from collections import Counter
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
OUT_DIR = Path("/content/drive/MyDrive/lexsi/grpo_diag")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "live_rollouts.json"
SUMMARY = OUT_DIR / "live_rollouts_summary.json"

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
os.environ["LEXSI_ALLOW_ANY_GPU"] = "1"

REPO = Path("/content/Hackathons")
if not (REPO / "lexsi").exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1", "-b",
        "cursor/lexsi-colab-rollout-diagnostics-8ccb",
        "https://github.com/AriCommits/Hackathons.git", str(REPO),
    ])
os.chdir(REPO / "lexsi")
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-e", ".[train]"])

ADAPTER = Path("/content/drive/MyDrive/lexsi/e2-checkpoint-150")
cmd = [
    sys.executable, "scripts/diagnose_grpo_rollouts.py", "--live-rollouts",
    "--allow-any-gpu", "--count", "4", "--output", str(OUT),
]
if ADAPTER.is_dir():
    cmd.extend(["--adapter", str(ADAPTER)])

print("STARTING unattended dump; safe to close laptop after this line prints.", flush=True)
subprocess.check_call(cmd)
payload = json.loads(OUT.read_text())
reasons = Counter(row["completion_reason"] for row in payload["rollouts"])
summary = {
    "output": str(OUT),
    "completion_reason_counts": dict(reasons),
    "rewards": [row["ledger"]["reward"] for row in payload["rollouts"]],
    "invalid_attempts": [row["ledger"]["invalid_attempts"] for row in payload["rollouts"]],
    "adapter": str(ADAPTER) if ADAPTER.is_dir() else None,
    "truncation_stats": payload.get("truncation_stats"),
}
SUMMARY.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
print("DONE", summary)
print("Forward this Counter later:", reasons)

## What still needs the Windows dual-GPU lab

- Full GRPO study with the pinned 5060 Ti UUID and protected `results/rl_v2/*` dirs
- Official memory probe / dual-card isolation claims
- Anything reported as study evidence (not diagnostic)